In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\PC\OneDrive - National Economics University\Máy tính\Fraud-Detection


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from src.config import PROCESSED_DATA_DIR
from src.train import (
    load_model_data,
    split_by_year,
    prepare_xy,
    train_all_models,
    tune_threshold,
    get_model_predictions,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

In [3]:
df = load_model_data(PROCESSED_DATA_DIR / "model_data.xlsx")

print("Shape:", df.shape)
display(df.head())

Shape: (1906, 15)


,CP,Năm,ROA,ROE,Net_Profit_Margin,Receivables_to_Revenue,CFO_to_Assets,SGAI,Revenue_Growth,CFO_to_Revenue,Debt_to_Assets,Soft_Assets,Debt_to_Equity,RSST_Accruals,Fraud
0,VGI,2018,-0.018295,-0.042918,-0.062762,0.413736,0.083650,0.959822,-0.113346,0.286968,0.573722,0.707667,1.345887,0.081258,1
1,AGM,2018,0.049644,0.072620,0.013214,0.027918,0.192490,1.078348,-0.080158,0.051235,0.316386,0.702721,0.462813,0.178462,0
2,BKC,2018,-0.030459,-0.063284,-0.049804,0.128463,-0.119212,0.965435,-0.012514,-0.194930,0.518703,0.692532,1.077719,0.024237,0
3,VGS,2018,0.031212,0.070713,0.006405,0.087645,0.207064,0.717813,0.157163,0.042492,0.558614,0.906885,1.265589,0.020757,0
4,VNG,2018,0.025293,0.044905,0.050892,0.514868,0.061342,1.122113,-0.006888,0.123425,0.436737,0.452033,0.775369,0.014201,0


In [4]:
train_df, valid_df, test_df = split_by_year(df)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Test shape :", test_df.shape)

print("\nFraud rate by split:")
print("Train:", round(train_df["Fraud"].mean(), 4))
print("Valid:", round(valid_df["Fraud"].mean(), 4))
print("Test :", round(test_df["Fraud"].mean(), 4))

Train shape: (1246, 15)
Valid shape: (331, 15)
Test shape : (329, 15)

Fraud rate by split:
Train: 0.2825
Valid: 0.287
Test : 0.2766


In [5]:
X_train, y_train, feature_cols = prepare_xy(train_df)
X_valid, y_valid, _ = prepare_xy(valid_df)
X_test, y_test, _ = prepare_xy(test_df)

print("Number of features:", len(feature_cols))
print(feature_cols)

print("\nTraining target distribution:")
display(y_train.value_counts(dropna=False))

Number of features: 12
['ROA', 'ROE', 'Net_Profit_Margin', 'Receivables_to_Revenue', 'CFO_to_Assets', 'SGAI', 'Revenue_Growth', 'CFO_to_Revenue', 'Debt_to_Assets', 'Soft_Assets', 'Debt_to_Equity', 'RSST_Accruals']

Training target distribution:


Fraud
0    894
1    352
Name: count, dtype: int64

In [6]:
artifacts = train_all_models(df)

print("Best model on test:", artifacts["best_model_name"])

Best model on test: SVM


In [7]:
print("Validation summary:")
display(artifacts["validation_summary"].round(4))

print("Test summary:")
display(artifacts["test_summary"].round(4))

Validation summary:


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,Split
0,SVM,0.5619,0.3656,0.7158,0.4840,0.6798,0.4484,Validation
1,LightGBM,0.6344,0.4085,0.6105,0.4895,0.6698,0.4312,Validation
2,XGBoost,0.6647,0.4355,0.5684,0.4932,0.6738,0.4390,Validation
3,ANN,0.7100,0.4949,0.5158,0.5052,0.6728,0.4434,Validation
4,Logistic Regression,0.6767,0.3333,0.1263,0.1832,0.6559,0.3912,Validation


Test summary:


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,Split
0,SVM,0.5775,0.3723,0.7692,0.5018,0.6984,0.4444,Test
1,XGBoost,0.6687,0.4392,0.7143,0.5439,0.6967,0.4663,Test
2,LightGBM,0.6474,0.4161,0.6813,0.5167,0.6965,0.4692,Test
3,ANN,0.6778,0.4051,0.3516,0.3765,0.6610,0.3813,Test
4,Logistic Regression,0.7143,0.4444,0.1319,0.2034,0.6682,0.4032,Test


In [8]:
print("Classification report:")
print(artifacts["classification_report"])

Classification report:
              precision    recall  f1-score   support

           0     0.8511    0.5042    0.6332       238
           1     0.3723    0.7692    0.5018        91

    accuracy                         0.5775       329
   macro avg     0.6117    0.6367    0.5675       329
weighted avg     0.7187    0.5775    0.5969       329



In [9]:
best_model_name = artifacts["validation_summary"].loc[0, "Model"]
print("Best model on validation:", best_model_name)

trained = artifacts["trained"]

valid_model_objects = {
    "Logistic Regression": (
        trained["logistic_regression"]["best_model"],
        trained["logistic_regression"]["use_scaled"],
    ),
    "XGBoost": (
        trained["xgboost"]["best_model"],
        trained["xgboost"]["use_scaled"],
    ),
    "LightGBM": (
        trained["lightgbm"]["best_model"],
        trained["lightgbm"]["use_scaled"],
    ),
    "ANN": (
        trained["ann"]["best_model"],
        trained["ann"]["use_scaled"],
    ),
    "SVM": (
        trained["svm"]["best_model"],
        trained["svm"]["use_scaled"],
    ),
}

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

model_obj, use_scaled = valid_model_objects[best_model_name]
X_valid_input = X_valid_scaled if use_scaled else X_valid

_, valid_prob = get_model_predictions(model_obj, X_valid_input)

threshold_df, best_threshold_row = tune_threshold(y_valid, valid_prob, metric="f1")

display(threshold_df.sort_values("f1", ascending=False))
print("Best threshold:", best_threshold_row["threshold"])

Best model on validation: SVM


,threshold,precision,recall,f1
5,0.35,0.428571,0.631579,0.510638
4,0.30,0.398810,0.705263,0.509506
6,0.40,0.447154,0.578947,0.504587
3,0.25,0.363636,0.715789,0.482270
2,0.20,0.350962,0.768421,0.481848
1,0.15,0.333333,0.842105,0.477612
7,0.45,0.460784,0.494737,0.477157
0,0.10,0.316726,0.936842,0.473404
8,0.50,0.520000,0.273684,0.358621
9,0.55,0.500000,0.084211,0.144144


Best threshold: 0.3500000000000001
